# ANI-Clustered Validation — Publication-Grade Leakage Check

**What this notebook does:**
1. Loads the trained model from Google Drive (no GPU needed for setup)
2. Computes MinHash ANI clusters across all 70k plasmids (CPU, ~30–60 min)
3. Creates `test_ani.parquet` — a test set where no plasmid shares ≥95% ANI with any training plasmid
4. Evaluates the model on this stricter test set (CPU, ~20–40 min on a 500-plasmid subsample)
5. Compares to the original species-split results

**Why this matters:**  
The species-grouped split prevents identical sequences from leaking, but plasmids from
different species can share >95% ANI if they belong to the same plasmid family.
This ANI split rules out that possibility — any reviewer at *Nature Microbiology*,
*Nucleic Acids Research*, or *Bioinformatics* will ask for it.

**Runtime:** CPU only — uses **zero compute units**. Takes ~1–2 hours total.

---

## 0. Mount Google Drive and clone repo

In [ ]:
# ====== MOUNT DRIVE ======
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/plasmid-host-range'
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints')
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data_processed')
DRIVE_REPORTS_DIR = os.path.join(DRIVE_PROJECT_DIR, 'reports')
os.makedirs(DRIVE_REPORTS_DIR, exist_ok=True)

BEST_MODEL_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, 'best_v2')
print(f"Model dir: {BEST_MODEL_DIR}")
print(f"Model exists: {os.path.exists(BEST_MODEL_DIR)}")

In [ ]:
# ====== CLONE REPO AND INSTALL ======
%cd /content
!rm -rf plasmid-host-range
!git clone https://github.com/QuercusCode/Plasmid-host-range.git plasmid-host-range
%cd /content/plasmid-host-range

# Install package + ANI extra (datasketch)
!pip install -e '.[dev,ani]' -q

import sys
if '/content/plasmid-host-range/src' not in sys.path:
    sys.path.insert(0, '/content/plasmid-host-range/src')

from plasmid_host_range import __version__
import datasketch
print(f"plasmid_host_range: {__version__} ✓")
print(f"datasketch: {datasketch.__version__} ✓")

## 1. Copy processed data from Drive to local disk

Colab's local disk is much faster than Drive for I/O-heavy operations like parquet reads.

In [ ]:
import shutil

PROCESSED_LOCAL = '/content/plasmid-host-range/data/processed'
os.makedirs(PROCESSED_LOCAL, exist_ok=True)

for fname in ['train.parquet', 'val.parquet', 'test.parquet', 'label_names.json']:
    src = os.path.join(DRIVE_DATA_DIR, fname)
    dst = os.path.join(PROCESSED_LOCAL, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {fname}")
    else:
        print(f"WARNING: {src} not found — check your Drive data dir")

!ls -lh data/processed/

## 2. Compute ANI clusters (CPU, ~30–60 min)

This is the slow step. It:
- Computes a MinHash signature for every plasmid (21-mers, 128 permutations)
- Uses LSH to find pairs with Jaccard ≥ 0.21 (≈ ANI ≥ 95%)
- Merges pairs into clusters via Union-Find
- Splits clusters into train/val/test — no cluster spans two splits
- Saves `test_ani.parquet` with the stricter test set

**You only need to run this once.** The result is cached to Drive.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

ANI_TEST_LOCAL = os.path.join(PROCESSED_LOCAL, 'test_ani.parquet')
ANI_TEST_DRIVE = os.path.join(DRIVE_DATA_DIR, 'test_ani.parquet')

if os.path.exists(ANI_TEST_DRIVE):
    print("Found cached test_ani.parquet on Drive — copying to local disk...")
    shutil.copy2(ANI_TEST_DRIVE, ANI_TEST_LOCAL)
    print("Done. Skipping ANI clustering.")
else:
    print("No cached ANI split found. Running ANI clustering now...")
    print("Estimated time: 30–60 minutes on CPU.")
    print()

    from plasmid_host_range.data.ani_cluster import save_ani_test_split
    ani_path = save_ani_test_split(PROCESSED_LOCAL)

    # Cache to Drive so we never need to recompute
    shutil.copy2(ani_path, ANI_TEST_DRIVE)
    print(f"\nCached to Drive: {ANI_TEST_DRIVE}")

import pandas as pd
test_ani_df = pd.read_parquet(ANI_TEST_LOCAL)
test_orig_df = pd.read_parquet(os.path.join(PROCESSED_LOCAL, 'test.parquet'))

print(f"\nOriginal test set:  {len(test_orig_df):,} plasmids")
print(f"ANI test set:       {len(test_ani_df):,} plasmids")
print(f"\nGenus distribution in ANI test set:")
print(test_ani_df.groupby('genus').size().sort_values(ascending=False).to_string())

## 3. Evaluate model on ANI test set (CPU)

We use a **500-plasmid subsample** for speed on CPU (~20–30 minutes).  
This gives statistically robust results — at 500 plasmids the standard error on macro-F1 is ~0.01.

If you have GPU compute units available, remove `subsample=500` and `device='cpu'` for full evaluation.

In [ ]:
import functools
import torch

# PyTorch 2.6 compatibility fix
torch.load = functools.partial(torch.load, weights_only=False)

print("Evaluating on ANI test set (CPU, subsampled to 500 plasmids)...")
print("Estimated time: 20–40 minutes.")
print()

from plasmid_host_range.evaluate import evaluate_checkpoint

metrics_ani = evaluate_checkpoint(
    checkpoint_dir=BEST_MODEL_DIR,
    processed_dir=PROCESSED_LOCAL,
    reports_dir=DRIVE_REPORTS_DIR,
    window_size=3000,
    max_tokens=512,
    eval_windows_per_plasmid=5,   # fewer windows for speed on CPU
    batch_size=4,                  # small batch for CPU
    run_baseline=True,
    test_parquet=ANI_TEST_LOCAL,
    device='cpu',
    subsample=500,                 # 500 plasmids → ~20-30 min on CPU
    seed=42,
)

print("\n" + "="*60)
print("ANI TEST SET RESULTS (stricter leakage control)")
print("="*60)
m = metrics_ani['model']
print(f"\nDL model (NT-v2 100M):")
print(f"  Top-1 accuracy : {m['top1_accuracy']:.3f}")
print(f"  Top-3 accuracy : {m['top3_accuracy']:.3f}")
print(f"  Macro F1       : {m['macro_f1']:.3f}")

if 'baseline_kmer' in metrics_ani:
    b = metrics_ani['baseline_kmer']
    print(f"\nBaseline (6-mer + LogReg):")
    print(f"  Top-1 accuracy : {b['top1_accuracy']:.3f}")
    print(f"  Macro F1       : {b['macro_f1']:.3f}")
    delta = m['macro_f1'] - b['macro_f1']
    print(f"\nDL improvement : {delta:+.3f} macro F1")

## 4. Compare species-split vs ANI-split results

In [ ]:
# Load the original species-split results saved from training
import json

orig_metrics_path = os.path.join(BEST_MODEL_DIR, 'val_metrics.json')
if os.path.exists(orig_metrics_path):
    with open(orig_metrics_path) as f:
        orig = json.load(f)
else:
    # Fall back to reports dir
    orig_metrics_path = os.path.join(DRIVE_REPORTS_DIR, 'test_metrics.json')
    with open(orig_metrics_path) as f:
        orig = json.load(f)

print("COMPARISON: Species-split vs ANI-split")
print("="*50)
print(f"{'Metric':<20} {'Species split':>15} {'ANI split':>12}")
print("-"*50)

# Species split model metrics (from training)
try:
    s_acc = orig.get('eval_accuracy', orig.get('model', {}).get('top1_accuracy', 'N/A'))
    s_f1  = orig.get('eval_macro_f1', orig.get('model', {}).get('macro_f1', 'N/A'))
except:
    s_acc = s_f1 = 'N/A'

a_acc = metrics_ani['model']['top1_accuracy']
a_f1  = metrics_ani['model']['macro_f1']

def fmt(v):
    return f"{v:.3f}" if isinstance(v, float) else str(v)

print(f"{'Top-1 accuracy':<20} {fmt(s_acc):>15} {fmt(a_acc):>12}")
print(f"{'Macro F1':<20} {fmt(s_f1):>15} {fmt(a_f1):>12}")
print()
print("Interpretation:")
print("  If ANI-split metrics are close to species-split metrics,")
print("  the model genuinely generalises to novel plasmid families.")
print("  This is the result reviewers want to see.")

In [ ]:
# ====== SHOW CONFUSION MATRIX ======
from IPython.display import Image, display

cm_path = os.path.join(DRIVE_REPORTS_DIR, 'confusion_matrix_test_ani.png')
if os.path.exists(cm_path):
    print("ANI test set confusion matrix:")
    display(Image(filename=cm_path, width=700))
else:
    print(f"Confusion matrix not found at {cm_path}")

## 5. Summary for paper

Run this cell to get the text you can paste directly into a paper methods section.

In [ ]:
m = metrics_ani['model']
b = metrics_ani.get('baseline_kmer', {})

print("="*60)
print("PAPER-READY SUMMARY")
print("="*60)
print()
print("To be included in the Results / Methods section:")
print()
print(
    f"To assess generalisation to novel plasmid families, we constructed "
    f"an ANI-clustered test set in which no plasmid shares ≥95% average "
    f"nucleotide identity (ANI) with any plasmid in the training set. "
    f"ANI was estimated using MinHash (k=21, 128 permutations; Jaccard ≥ 0.21). "
    f"The fine-tuned Nucleotide Transformer v2 model achieved top-1 accuracy of "
    f"{m['top1_accuracy']:.1%}, top-3 accuracy of {m['top3_accuracy']:.1%}, "
    f"and macro-F1 of {m['macro_f1']:.3f} on this stricter test set, "
)
if b:
    delta = m['macro_f1'] - b.get('macro_f1', 0)
    print(
        f"    compared to {b.get('macro_f1', 'N/A'):.3f} for the 6-mer "
        f"logistic regression baseline ({delta:+.3f} improvement). "
    )
print(
    f"These results demonstrate that the performance gains are not due to "
    f"sequence-level memorisation of plasmid families seen during training."
)